# Embedding Validation
This notebook tests the `FeatureProcessor` from `src/embedder.py` on the new Universal Training Table.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add src to path
sys.path.append('../')
from src.embedder import FeatureProcessor

processor = FeatureProcessor()

/Users/ryanfue/Library/Caches/pypoetry/virtualenvs/smartcomps-16436_SS-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10451.51it/s]


## 1. Load Universal Training Table

In [2]:
# Load data
data_path = "../data/processed/UNIVERSAL_training.parquet"

df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} rows")
df.head()

Loaded 6788 rows


,ticker,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,employee_count,estimated_revenue,sector,industry,business_summary
0,NVDA,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,42000.0,2.534910e+11,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...
1,GOOGL,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,194668.0,4.224980e+11,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...
2,AAPL,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,166000.0,4.514420e+11,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ..."
3,MSFT,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,228000.0,3.182730e+11,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...
4,AMZN,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,1575000.0,7.427760e+11,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of..."


## 2. Test Embedding Generation
Using `all-MiniLM-L6-v2` local model.

In [3]:
df_emb = processor.embed_summaries(df.head(10)) # Small sample for validation
df_emb.head()

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]


,ticker,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,employee_count,estimated_revenue,sector,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,NVDA,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,42000.0,2.534910e+11,Technology,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,GOOGL,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,194668.0,4.224980e+11,Communication Services,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,AAPL,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,166000.0,4.514420e+11,Technology,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,MSFT,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,228000.0,3.182730e+11,Technology,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,AMZN,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,1575000.0,7.427760e+11,Consumer Cyclical,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between first two companies to verify embedding semantic quality
sim = cosine_similarity([df_emb.iloc[0].filter(like='nlp_')], [df_emb.iloc[1].filter(like='nlp_')])
print(f"Similarity between {df_emb.iloc[0]['ticker']} and {df_emb.iloc[1]['ticker']}: {sim[0][0]:.4f}")

Similarity between NVDA and GOOGL: 0.4038


## 3. Test Feature Preservation & Metadata Dropping
Ensuring that metadata is removed but features like `sector` are preserved for the Valuation Engine.

In [5]:
# Demonstrate dropping metadata while keeping 'sector'
df_ml = processor.drop_extra_columns(df_emb)

print(f"Columns after dropping metadata: {df_ml.columns.tolist()[:10]}...")
print(f"Sector preserved: {'sector' in df_ml.columns}")
print(f"Remaining columns: {len(df_ml.columns)}")

df_ml.head()

Columns after dropping metadata: ['enterprise_value', 'forwardPE', 'ev_to_ebitda', 'ebitda', 'total_cash', 'total_debt', 'employee_count', 'estimated_revenue', 'sector', 'nlp_0']...
Sector preserved: True
Remaining columns: 393


,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,employee_count,estimated_revenue,sector,nlp_0,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,42000.0,2.534910e+11,Technology,-0.059150,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,194668.0,4.224980e+11,Communication Services,-0.059735,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,166000.0,4.514420e+11,Technology,-0.042323,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,228000.0,3.182730e+11,Technology,-0.014270,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,1575000.0,7.427760e+11,Consumer Cyclical,0.039800,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085


## 4. Verify Final Universal Embedded Dataset
Verify the output of the full pipeline.

In [7]:
universal_embedded = "../data/processed/UNIVERSAL_embedded.parquet"

if os.path.exists(universal_embedded):
    df_final = pd.read_parquet(universal_embedded)
    print(f"Universal Embedded Table Shape: {df_final.shape}")
    print(f"Critical features present: {'sector' in df_final.columns and 'nlp_0' in df_final.columns}")
    display(df_final.head(2))
else:
    print("Universal embedded file not found. Run 'poetry run python src/embedder.py' first.")

Universal Embedded Table Shape: (6788, 393)
Critical features present: True


,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,employee_count,estimated_revenue,sector,nlp_0,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,42000.0,2.534910e+11,Technology,-0.059150,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,194668.0,4.224980e+11,Communication Services,-0.059735,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
